# Лабораторная работа 1.2. Метод прогонки

**Задание.** Реализовать метод прогонки в виде программы, задавая в качестве входных данных
ненулевые элементы матрицы системы и вектор правых частей. Используя разработанное ПО,
решить СЛАУ с трёхдиагональной матрицей.

$$
\begin{cases}
8x_1 - 2x_2 = -14 \\
7x_1 - 19x_2 + 9x_3 = -55 \\
-4x_2 + 21x_3 - 8x_4 = 49 \\
7x_3 - 23x_4 + 9x_5 = 86 \\
4x_4 - 7x_5 = 8
\end{cases}
$$

## Теория

Трёхдиагональная система записывается в виде

$$a_i x_{i-1} + b_i x_i + c_i x_{i+1} = d_i, \qquad i = 1..n, \qquad a_1 = 0,\ c_n = 0,$$

где $a_i$ — поддиагональ, $b_i$ — главная диагональ, $c_i$ — наддиагональ.
Поэтому на вход подаются только три вектора $a, b, c$ и правая часть $d$ — всего $3n-2$ ненулевых элемента матрицы
вместо $n^2$.

Решение ищется в виде $x_i = P_i x_{i+1} + Q_i$.

**Прямой ход** — вычисление прогоночных коэффициентов:

$$P_1 = \frac{-c_1}{b_1},\quad Q_1 = \frac{d_1}{b_1}; \qquad
P_i = \frac{-c_i}{b_i + a_i P_{i-1}},\quad Q_i = \frac{d_i - a_i Q_{i-1}}{b_i + a_i P_{i-1}},\quad i = 2..n.$$

**Обратный ход:** $x_n = Q_n$ (так как $c_n = 0 \Rightarrow P_n = 0$), $\; x_i = P_i x_{i+1} + Q_i,\ i = n-1..1$.

**Достаточное условие корректности и устойчивости** — диагональное преобладание:
$|b_i| \ge |a_i| + |c_i|$, причём хотя бы для одного $i$ неравенство строгое. Тогда $|P_i| \le 1$
и знаменатели не обращаются в ноль.

## Реализация

In [1]:
def check_tridiagonal(a, b, c):
    """Проверка достаточного условия устойчивости прогонки (диагональное преобладание)."""
    n = len(b)
    ok, strict = True, False
    for i in range(n):
        side = abs(a[i]) + abs(c[i])
        print(f"  i={i + 1}: |b| = {abs(b[i]):>4}  |a|+|c| = {side:>4}  ->", "ok" if abs(b[i]) >= side else "НАРУШЕНО")
        ok &= abs(b[i]) >= side
        strict |= abs(b[i]) > side
    return ok and strict


def tridiagonal_solve(a, b, c, d):
    """Метод прогонки.

    a — поддиагональ (a[0] = 0), b — главная диагональ,
    c — наддиагональ (c[-1] = 0), d — правая часть.
    Возвращает (x, P, Q).
    """
    n = len(b)
    if not (len(a) == len(c) == len(d) == n):
        raise ValueError("Векторы a, b, c, d должны иметь одинаковую длину")
    if a[0] != 0 or c[-1] != 0:
        raise ValueError("Должно выполняться a[0] = 0 и c[n-1] = 0")

    P, Q = [0.0] * n, [0.0] * n
    # прямой ход
    for i in range(n):
        denom = b[i] + (a[i] * P[i - 1] if i > 0 else 0.0)
        if abs(denom) < 1e-14:
            raise ZeroDivisionError(f"Нулевой знаменатель на шаге {i + 1}")
        P[i] = -c[i] / denom
        Q[i] = (d[i] - (a[i] * Q[i - 1] if i > 0 else 0.0)) / denom

    # обратный ход
    x = [0.0] * n
    x[-1] = Q[-1]
    for i in range(n - 2, -1, -1):
        x[i] = P[i] * x[i + 1] + Q[i]
    return x, P, Q


def tridiagonal_to_full(a, b, c):
    """Полная матрица — только для проверки и вывода."""
    n = len(b)
    A = [[0.0] * n for _ in range(n)]
    for i in range(n):
        A[i][i] = b[i]
        if i > 0:
            A[i][i - 1] = a[i]
        if i < n - 1:
            A[i][i + 1] = c[i]
    return A

## Исходные данные (только ненулевые элементы)

In [2]:
a = [ 0,   7,  -4,   7,  4]   # поддиагональ
b = [ 8, -19,  21, -23, -7]   # главная диагональ
c = [-2,   9,  -8,   9,  0]   # наддиагональ
d = [-14, -55, 49,  86,  8]   # правая часть

A = tridiagonal_to_full(a, b, c)
print("Матрица системы (восстановлена для наглядности):")
for row, di in zip(A, d):
    print("  [" + " ".join(f"{v:5.0f}" for v in row) + f" ]   | {di:5}")

Матрица системы (восстановлена для наглядности):
  [    8    -2     0     0     0 ]   |   -14
  [    7   -19     9     0     0 ]   |   -55
  [    0    -4    21    -8     0 ]   |    49
  [    0     0     7   -23     9 ]   |    86
  [    0     0     0     4    -7 ]   |     8


## Проверка условия устойчивости

In [3]:
print("Диагональное преобладание:", "выполнено" if check_tridiagonal(a, b, c) else "НЕ выполнено")

  i=1: |b| =    8  |a|+|c| =    2  -> ok
  i=2: |b| =   19  |a|+|c| =   16  -> ok
  i=3: |b| =   21  |a|+|c| =   12  -> ok
  i=4: |b| =   23  |a|+|c| =   16  -> ok
  i=5: |b| =    7  |a|+|c| =    4  -> ok
Диагональное преобладание: выполнено


## Решение

In [4]:
x, P, Q = tridiagonal_solve(a, b, c, d)

print(" i        P_i          Q_i")
for i, (p, q) in enumerate(zip(P, Q), 1):
    print(f"{i:2}  {p + 0.0:11.6f}  {q + 0.0:11.6f}")  # +0.0 убирает «-0.0»
print()
for i, xi in enumerate(x, 1):
    print(f"x{i} = {xi:.6f}")

r = [sum(A[i][j] * x[j] for j in range(len(x))) - d[i] for i in range(len(x))]
print("\nНевязка Ax - d:", [f"{v:.2e}" for v in r])

 i        P_i          Q_i
 1     0.250000    -1.750000
 2     0.521739     2.478261
 3     0.422989     3.114943
 4     0.449122    -3.203510
 5     0.000000    -4.000000

x1 = -1.000000
x2 = 3.000000
x3 = 1.000000
x4 = -5.000000
x5 = -4.000000

Невязка Ax - d: ['0.00e+00', '0.00e+00', '-7.11e-15', '0.00e+00', '0.00e+00']


## Проверка с помощью NumPy (необязательно)

In [5]:
try:
    import numpy as np
    x_np = np.linalg.solve(np.array(A), np.array(d, float))
    print("x (numpy):", x_np)
    print("max |x - x_numpy| =", np.abs(np.array(x) - x_np).max())
except ImportError:
    print("NumPy не установлен — пропускаем сверку")

x (numpy): [-1.  3.  1. -5. -4.]
max |x - x_numpy| = 4.440892098500626e-16


## Вывод

Реализован метод прогонки, принимающий на вход только ненулевые элементы трёхдиагональной матрицы
(векторы $a, b, c$) и правую часть $d$. Матрица системы обладает строгим диагональным преобладанием,
поэтому все $|P_i| \le 1$ и метод устойчив. Решение получено за $O(n)$ операций; невязка и сверка
с NumPy подтверждают его корректность.